# Diabetic Retinopathy Stage Detection Pipeline

This notebook serves as the main entry point to execute the Diabetic Retinopathy classification pipeline on Google Colab.

## 1. Setup and Clone Repo
Clone the repository and install the necessary requirements.

In [ ]:
# Clone the repository
REPO_NAME = "Computer-Vision-CW---diabetic-retinopathy-stage-detection"
!git clone https://github.com/ErandiPathirana/Computer-Vision-CW---diabetic-retinopathy-stage-detection.git
%cd {REPO_NAME}

!pip install -r requirements.txt -q


## 2. Kaggle Download
Downloads the APTOS 2019 Blindness Detection dataset using the Kaggle API.
*Make sure you have added your `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Colab Secrets tab on the left.*

In [ ]:
# ============================================================
# Dataset Setup — APTOS 2019 Blindness Detection
# ============================================================
# Three supported methods (tried in order):
#   1. Kaggle API   — add KAGGLE_USERNAME + KAGGLE_KEY to Colab Secrets
#   2. Manual upload — run the upload widget below, then re-run this cell
#   3. Google Drive  — set DRIVE_ZIP_PATH to your zip's Drive path
#
# All paths come from src/config.py — do not hard-code paths here.
# ============================================================

import os, glob, zipfile, shutil
from src.config import RAW_DATA_DIR

# --- Option 3: set this to your Drive path if you prefer Drive ---
DRIVE_ZIP_PATH = ""  # e.g. "/content/drive/MyDrive/aptos2019-blindness-detection.zip"

def _unzip(zip_path, dest):
    """Extract zip and flatten any single top-level folder."""
    print(f"Extracting {zip_path} → {dest} ...")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest)
    # If extraction created a single subdirectory, hoist its contents up
    entries = os.listdir(dest)
    if len(entries) == 1 and os.path.isdir(os.path.join(dest, entries[0])):
        sub = os.path.join(dest, entries[0])
        for item in os.listdir(sub):
            shutil.move(os.path.join(sub, item), dest)
        os.rmdir(sub)
    print("Extraction complete.")

# Skip if already extracted
train_csv = os.path.join(RAW_DATA_DIR, 'train.csv')
if os.path.exists(train_csv):
    print(f"Dataset already present at {RAW_DATA_DIR}. Skipping download.")
else:
    zip_path = None

    # --- Method 1: Kaggle API ---
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        import subprocess
        os.makedirs(RAW_DATA_DIR, exist_ok=True)
        result = subprocess.run(
            ["kaggle", "competitions", "download",
             "-c", "aptos2019-blindness-detection",
             "-p", RAW_DATA_DIR],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            zips = glob.glob(os.path.join(RAW_DATA_DIR, '*.zip'))
            if zips:
                zip_path = zips[0]
                print("Kaggle API download successful.")
            else:
                print("Kaggle download ran but no zip found; trying next method.")
        else:
            print("Kaggle API failed:", result.stderr.strip())
    except Exception as e:
        print(f"Kaggle API unavailable: {e}")

    # --- Method 2: Manual upload (already uploaded zip in /content/) ---
    if zip_path is None:
        uploaded_zips = glob.glob('/content/*.zip')
        if uploaded_zips:
            zip_path = uploaded_zips[0]
            print(f"Found manually uploaded zip: {zip_path}")

    # --- Method 3: Google Drive ---
    if zip_path is None and DRIVE_ZIP_PATH and os.path.exists(DRIVE_ZIP_PATH):
        zip_path = DRIVE_ZIP_PATH
        print(f"Using Drive zip: {zip_path}")

    if zip_path is None:
        raise FileNotFoundError(
            "Dataset zip not found. Please do ONE of the following:\n"
            "  A) Add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets and re-run.\n"
            "  B) Upload the APTOS zip manually (Files panel → Upload), then re-run.\n"
            "  C) Mount Google Drive and set DRIVE_ZIP_PATH above, then re-run."
        )

    _unzip(zip_path, RAW_DATA_DIR)
    # Clean up the zip to save disk space
    if zip_path.startswith(RAW_DATA_DIR):
        os.remove(zip_path)
        print("Zip removed to save space.")

print(f"\nDataset root: {RAW_DATA_DIR}")
print("Contents:", os.listdir(RAW_DATA_DIR))


## 3. Exploratory Data Analysis (EDA)
Analyzes the class distributions and image properties.

In [ ]:
from src.data import run_eda
run_eda()

## 4. Split
Performs a strict 70/15/15 stratified train/validation/test split to prevent data leakage.

In [ ]:
from src.data_loader import split_dataset
split_dataset()

## 5. Preprocessing
Executes cropping, CLAHE, and unsharp masking, caching the processed images to disk.

In [ ]:
import os
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.preprocess import process_and_cache_dataset, generate_comparison_figures
from glob import glob
import random

train_input = os.path.join(RAW_DATA_DIR, "train_images")
train_output = os.path.join(PROCESSED_DATA_DIR, "train_images")
process_and_cache_dataset(train_input, train_output)

test_input = os.path.join(RAW_DATA_DIR, "test_images")
test_output = os.path.join(PROCESSED_DATA_DIR, "test_images")
if os.path.exists(test_input):
    process_and_cache_dataset(test_input, test_output)

# Optional: Generate a few comparison figures to see the effect
sample_imgs = glob(os.path.join(train_input, "*.png"))
if len(sample_imgs) >= 3:
    random.seed(42)
    generate_comparison_figures(random.sample(sample_imgs, 3))

## 6. Augmentation and Imbalance
Visualizes the training data augmentations and the class weighting strategy used to combat imbalance.

In [ ]:
from src.augment import show_augmented_examples
from src.data import compute_class_weights_from_train, visualize_oversampling

show_augmented_examples()
class_weights = compute_class_weights_from_train()
visualize_oversampling()

## 7. Model Architecture
Constructs the backbone (EfficientNetB0) and displays the parameter shifts between training phases.

In [ ]:
from src.model import display_summary
display_summary()

## 8. Training
Executes the 2-phase training loop. 
*Note: Adjust epochs appropriately for final evaluation.*

In [ ]:
from src.train import train_pipeline

# Set epochs to a lower value for quick testing, or higher (15, 20) for final results
train_pipeline(epochs_phase1=10, epochs_phase2=15)

## 9. Evaluation
Evaluates the best model strictly on the untouched test set.

In [ ]:
from src.evaluate import evaluate_model
evaluate_model()

## 9b. Saved Metrics (JSON)
Displays the `metrics.json` and `class_names.json` files written by `evaluate_model()`.

In [ ]:
import json, os
from src.config import MODELS_DIR

metrics_path = os.path.join(MODELS_DIR, 'metrics.json')
class_names_path = os.path.join(MODELS_DIR, 'class_names.json')

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('=== metrics.json ===')
    print(f"  Accuracy              : {metrics['accuracy']:.4f}")
    print(f"  Macro Precision       : {metrics['macro_precision']:.4f}")
    print(f"  Macro Recall          : {metrics['macro_recall']:.4f}")
    print(f"  Macro F1              : {metrics['macro_f1']:.4f}")
    print(f"  Quadratic Weighted K  : {metrics['quadratic_weighted_kappa']:.4f}")
else:
    print(f'metrics.json not found at {metrics_path}. Run evaluate_model() first.')

if os.path.exists(class_names_path):
    with open(class_names_path) as f:
        print('\n=== class_names.json ===')
        print(json.load(f))
else:
    print(f'class_names.json not found at {class_names_path}.')


## 10. Error Analysis
Displays the generated summary of where the model failed and why, alongside Grad-CAM interpretations of incorrect predictions.

In [ ]:
from IPython.display import display, Markdown, Image
from src.config import RESULTS_DIR
import os

error_md = os.path.join(RESULTS_DIR, 'evaluation', 'error_analysis_summary.md')
if os.path.exists(error_md):
    with open(error_md, 'r') as f:
        display(Markdown(f.read()))

incorrect_cam = os.path.join(RESULTS_DIR, 'evaluation', 'gradcam_incorrect.png')
if os.path.exists(incorrect_cam):
    print('\n--- Grad-CAM: Incorrect Predictions ---')
    display(Image(incorrect_cam))


## 11. Sample Predictions
Displays Grad-CAM heatmaps for correctly classified test instances.

In [ ]:
correct_cam = os.path.join(RESULTS_DIR, "evaluation", "gradcam_correct.png")
if os.path.exists(correct_cam):
    print("\n--- Grad-CAM: Correct Predictions ---")
    display(Image(correct_cam))

## 12. Export Results
Zips the entire `results/` folder (including models, plots, logs, and CAMs) for download.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('results_archive', 'zip', 'results')
files.download('results_archive.zip')